# 02 — Dimensionality reduction on the monkey working-memory dataset (DANDI 000620) — delay-aligned

The **essential** dim-red, mirroring the mouse [`04_dimred_msr.ipynb`](04_dimred_msr.ipynb): **PCA**,
**dPCA**, **RRR** — with slow-rotating 3-D **PCA** and **dPCA** GIFs for slides. FEF + DMFC, one
session (**sub-Elgar**), **delay-aligned** (t = 0 = delay onset; see `07_monkey_hmm`).

**dPCA now runs.** The task has no stimulus×choice factorial, but the timestamps give one behavioural
factor — the animal's **response time** (reward onset − response onset), median-split short/long. dPCA
decomposes population variance into **time** (condition-independent) vs **response-time** vs their
interaction. Colours: cividis (time), Okabe-Ito categorical, marker *shape* = condition — CVD-safe.

## Part 0 — Setup

In [ ]:
import sys, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.monkey import load_monkey_aligned, PHASE_NAMES
from sklearn.decomposition import PCA
from PIL import Image as PILImage
from IPython.display import Image
from src.geometry.embed import population_matrix
from src.dynamics.rrr import rrr_cv_curve, within_area_control, saturation_rank
from src.dynamics.dpca import balanced_by_label, fit_dpca, variance_by_marginalization
from src.viz.anim3d import rotating_3d_gif
from src.viz.style import set_style, state_colors, savefig, figure_path, stroke, SEQUENTIAL, DIVERGING
set_style()

SESSION = "ses-2022-08-20"          # 43 FEF + 56 DMFC units; sub-Elgar
AREAS = ["FEF", "DMFC"]
SEED = 0
SMOOTH_MS = 100.0
PHASE = "monkey_dimred"

def show_gif_preview(gif_path, frame=30):
    """Save+show a small static frame of the GIF (embedding the full GIF bloats the notebook).
    The slide-ready animation is the .gif on disk."""
    im = PILImage.open(gif_path); im.seek(min(frame, im.n_frames - 1))
    prev = gif_path.with_name(gif_path.stem + '_preview.png')
    im.convert('RGB').save(prev)
    return Image(filename=str(prev))

## Part 1 — Load one session, delay-aligned

In [ ]:
mk = load_monkey_aligned(SESSION)          # delay-aligned, window [-1.4,+2.2]s @ 100 ms
bin_centers = mk["bin_centers"]
tensors = mk["tensors"]
events = mk["events"]                      # (n_trials, 7) event onsets, DELAY-relative (delay=0)
n_trials = mk["n_trials"]
BIN_S = mk["window"][2]

# phase-onset times (delay-relative medians): stimulus, delay(=0), cue, response, reward
ev_med = np.nanmedian(events, axis=0)
ONSETS = {"stimulus": ev_med[1], "delay": ev_med[2], "cue": ev_med[3],
          "response": ev_med[4], "reward": ev_med[5]}

def phase_lines(ax, top=1.03, shade=True):
    if shade:
        ax.axvspan(ONSETS["delay"], ONSETS["cue"], color="0.92", zorder=0)   # the WM delay period
    for name, t in ONSETS.items():
        bold = name == "delay"
        ax.axvline(t, color="black" if bold else "0.55", ls="--", lw=2.0 if bold else 1.0, zorder=1)
        ax.text(t, top, name, rotation=90, va="bottom", ha="center", fontsize=7,
                color="black" if bold else "0.45")

print(f"kept {n_trials} trials; {len(bin_centers)} bins; window "
      f"[{bin_centers[0]:.2f}, {bin_centers[-1]:.2f}] s from DELAY onset")
print("phase onsets (delay-relative):", {k: round(float(v), 2) for k, v in ONSETS.items()})
for a in AREAS:
    print(f"  {a}: {tensors[a].shape}  (trials, bins, units)")
time_of_row = np.tile(bin_centers, n_trials)

## Part 2 — PCA, and the rotating 3-D GIF

A point is one (trial, bin) population vector: `sqrt`, smoothed (σ = 100 ms), z-scored. The GIF shows
the time-coloured cloud plus the **trial-mean trajectory** (dark path).

In [ ]:
pca_panels = []
for a in AREAS:
    X = population_matrix(tensors[a], None, bin_size_s=BIN_S, smooth_sigma_ms=SMOOTH_MS)
    p = PCA(n_components=3, random_state=SEED).fit(X); Y = p.transform(X)
    evr = p.explained_variance_ratio_[:3] * 100
    mean_traj = Y.reshape(n_trials, len(bin_centers), 3).mean(axis=0)
    pca_panels.append({"series": [
        {"xyz": Y, "kind": "scatter", "c": time_of_row, "s": 6},
        {"xyz": mean_traj, "kind": "linemarker", "c": bin_centers, "s": 40, "line_color": "0.3", "lw": 2.2}],
        "axis_labels": (f"PC1 {evr[0]:.0f}%", f"PC2 {evr[1]:.0f}%", f"PC3 {evr[2]:.0f}%"), "title": a})
    print(f"{a}: PC1-3 variance = {evr.round(1)} % (sum {evr.sum():.0f} %)")
pca_gif = rotating_3d_gif(pca_panels, figure_path(PHASE, "rotate_pca", ext="gif"),
    suptitle="Monkey PCA (delay-aligned) — cloud + trial-mean, colour = time",
    colorbar_label="time from delay (s)", n_frames=90, fps=12, dpi=85)
print("PCA GIF ->", pca_gif, "(open the .gif for the rotating slide version)")
show_gif_preview(pca_gif)

## Part 3 — dPCA: time vs the animal's response time

The behavioural factor is the **response time** (reward onset − response onset), median-split into
short/long. dPCA (`balanced_by_label` → `fit_dpca(labels="ct")`) splits variance into **time (t)**,
**condition (c)** and their interaction. Expect **time to dominate** (a fixed-timing task); the point
is to *isolate* the small response-time component. The GIF plots two demixed axes vs the condition
axis, one trajectory per response-time group.

In [ ]:
t_resp = events[:, 5] - events[:, 4]            # reward onset - response onset = response time
cond = (t_resp > np.median(t_resp)).astype(int)   # 0 = short, 1 = long
SYM = {0: "o", 1: "D"}; LAB = {0: "response SHORT", 1: "response LONG"}
print(f"response time: median {np.median(t_resp):.3f} s -> short {int((cond==0).sum())} / long {int((cond==1).sum())}")

dpca_panels = []
for a in AREAS:
    X, trialX, Kc, levels = balanced_by_label(tensors[a], cond, seed=SEED)
    model, Z = fit_dpca(X, trialX, n_components=3, labels="ct", protect=("t",))
    var = variance_by_marginalization(model).set_index("marginalization")["pct_of_explained"]
    print(f"{a}: K={Kc}/cell  variance% -> time {var.get('t',0):.0f}, "
          f"response-time {var.get('c',0):.0f}, interaction {var.get('ct',0):.0f}")
    series = [{"xyz": np.column_stack([Z["t"][0, lv], Z["c"][0, lv], Z["t"][1, lv]]),
               "kind": "linemarker", "c": bin_centers, "s": 30, "marker": SYM[lv],
               "line_color": "0.8", "lw": 1.8} for lv in range(len(levels))]
    dpca_panels.append({"series": series,
        "axis_labels": (f"time-1 {var.get('t',0):.0f}%", f"response-time {var.get('c',0):.0f}%", "time-2"),
        "title": f"{a} (K={Kc}/cell)"})
dpca_gif = rotating_3d_gif(dpca_panels, figure_path(PHASE, "rotate_dpca", ext="gif"),
    suptitle="Monkey dPCA — time vs response-time (short/long) · colour = time",
    colorbar_label="time from delay (s)", legend=[(LAB[k], SYM[k]) for k in SYM],
    n_frames=90, fps=12, dpi=85)
print("dPCA GIF ->", dpca_gif, "(open the .gif for the rotating slide version)")
show_gif_preview(dpca_gif)

## Part 4 — RRR: the communication subspace between FEF and DMFC

Reduced-rank regression predicts one area from the other, restricted to rank *r*; the saturation rank
is the channel dimensionality (Semedo 2019). The reference is the **within-area control** (predict half
of an area from its other half). CV grouped by trial. No condition labels needed.

In [ ]:
def _flat_z(counts):
    X = counts.reshape(-1, counts.shape[2]).astype(float)
    return (X - X.mean(0)) / (X.std(0) + 1e-9)
groups = np.repeat(np.arange(n_trials), len(bin_centers))
Xf, Xd = _flat_z(tensors["FEF"]), _flat_z(tensors["DMFC"])
curves = {"FEF->DMFC": rrr_cv_curve(Xf, Xd, groups), "DMFC->FEF": rrr_cv_curve(Xd, Xf, groups)}
ctrl = {"DMFC self": within_area_control(Xd, groups, seed=SEED),
        "FEF self": within_area_control(Xf, groups, seed=SEED)}
fig, ax = plt.subplots(figsize=(7.2, 4.4))
for lab, c in curves.items():
    ax.errorbar(c["rank"], c["mean_r2"], yerr=c["sem_r2"], marker="o", lw=1.8, capsize=3,
                label=f"{lab}  (sat rank {saturation_rank(c)})", path_effects=stroke())
for lab, c in ctrl.items():
    ax.plot(c["rank"], c["mean_r2"], ls="--", lw=1.6, label=f"{lab}  (sat rank {saturation_rank(c)})")
ax.set(xlabel="rank", ylabel="held-out R^2", title=f"{SESSION}: RRR  FEF <-> DMFC")
ax.legend(frameon=False, fontsize=8); fig.tight_layout(); savefig(fig, PHASE, f"{SESSION}_rrr"); plt.show()
for lab, c in curves.items():
    print(f"{lab}: best R^2 = {c['mean_r2'].max():.3f}, saturation rank = {saturation_rank(c)}")

## Part 5 — Summary and next

- **PCA** (rotating GIF): FEF a coherent low-D trajectory, DMFC more distributed.
- **dPCA**: population variance is dominated by **time** (~94–95 %); the **response-time** component is
  small (~3–4 %) but isolated — the honest dPCA answer for a fixed-timing task (there is no
  stimulus×choice factorial, so response time is the one behavioural factor).
- **RRR**: cross-area prediction is weak and **asymmetric** — DMFC→FEF (R² ≈ 0.11) > FEF→DMFC (≈ 0.05);
  FEF is low-dimensional to itself, DMFC is not.

**Next:** replicate across the 14 usable sessions and pool; add the 2nd animal; if a stimulus/choice
label becomes available, run the mouse-style 2×2 dPCA for the sensory/decision split.